# 🧚 Portefee — Portfolio Visualization

Visualizes your stock & ETF portfolio with:
- **Per-stock/ETF charts** — market price evolution with buy-point annotations
- **Combined portfolio chart** — total market value vs. cost basis over time

Data sourced from your Notion "Stocks and ETFs" database.  
Market prices pulled from Yahoo Finance.

## 1. Setup & Install

In [ ]:
!pip install -q yfinance matplotlib pandas

In [ ]:
import os
import warnings
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import pandas as pd
import yfinance as yf

warnings.filterwarnings("ignore")

# Output directory (in Google Drive if mounted, otherwise local)
USE_DRIVE = False  # @param {type:"boolean"}

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/portefee/charts'
else:
    OUT_DIR = '/content/portefee_charts'

os.makedirs(OUT_DIR, exist_ok=True)
print(f"Charts will be saved to: {OUT_DIR}")

## 2. Portfolio Data

Ticker mapping and buy history from Notion.  
Update this section when you make new purchases.

In [ ]:
# ─── Yahoo Finance ticker mapping ───────────────────────────
TICKER_MAP = {
    "MSFT":   "MSFT",
    "GOOGL":  "GOOGL",
    "NVDA":   "NVDA",
    "AMZN":   "AMZN",
    "BEL20":  "BEL20.BR",
}

# ─── Currency per position ──────────────────────────────────
TICKER_CURRENCY = {
    "MSFT": "USD", "GOOGL": "USD", "NVDA": "USD", "AMZN": "USD",
    "BEL20": "EUR",
}

In [ ]:
# ─── Buy history ───────────────────────────────────────────
# Format: (date, num_shares, price_per_share, total_costs)
# Fictive list, provide your own
# Prices in native currency of the instrument

BUYS = {
    "MSFT": [
        ("2022-07-27", 5,  264.29, 22.04),
        ("2025-10-29", 15, 542.00, 0.00),
        ("2026-02-23", 4,  395.00, 5.55),
    ],
    "GOOGL": [
        ("2022-08-16", 15, 122.00, 25.95),
        ("2024-10-02", 16, 167.78, 16.55),
        ("2025-10-24", 12, 260.79, 18.33),
    ],
    "NVDA": [
        ("2024-12-27", 145, 138.51, 131.11),
        ("2025-08-28", 60,  180.93, 63.38),
        ("2025-09-29", 50,  181.62, 52.72),
    ],
    "AMZN": [
        ("2022-08-22", 15, 135.00, 28.02),
        ("2024-12-17", 13, 232.40, 19.59),
    ],
    "BEL20": [
        ("2025-12-05", 1, 76.16, 0.09),
        ("2026-01-05", 1, 75.95, 0.09),
        ("2026-02-05", 1, 81.87, 0.10),
    ],
}

## 3. Helper Functions

In [ ]:
def get_earliest_buy(buys_dict: dict) -> str:
    """Return the earliest buy date across all positions."""
    all_dates = [d for positions in buys_dict.values() for d, *_ in positions]
    return min(all_dates)


def download_price_history(yf_ticker: str, start: str) -> pd.DataFrame:
    """Download daily close prices from Yahoo Finance."""
    try:
        df = yf.download(yf_ticker, start=start, progress=False)
        if df.empty:
            return pd.DataFrame()
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        return df[["Close"]].rename(columns={"Close": "price"})
    except Exception as e:
        print(f"  ⚠️  Failed to download {yf_ticker}: {e}")
        return pd.DataFrame()


def compute_cumulative_shares(buy_list: list, date_index: pd.DatetimeIndex) -> pd.Series:
    """Given a list of buys, return cumulative shares held over time."""
    shares = pd.Series(0.0, index=date_index)
    for date_str, num, *_ in buy_list:
        dt = pd.Timestamp(date_str)
        shares[shares.index >= dt] += num
    return shares

## 4. Download Market Data

In [ ]:
earliest = get_earliest_buy(BUYS)
start_date = (pd.Timestamp(earliest) - timedelta(days=30)).strftime("%Y-%m-%d")

print(f"📅 Downloading price data from {start_date} …\n")

prices = {}
failed = []

for name, yf_ticker in TICKER_MAP.items():
    if name not in BUYS:
        continue
    print(f"  📈 {name} ({yf_ticker}) …", end=" ")
    df = download_price_history(yf_ticker, start_date)
    if not df.empty:
        prices[name] = df
        print(f"✅ {len(df)} days")
    else:
        failed.append(name)
        print("❌ FAILED")

print(f"\n✅ Downloaded {len(prices)}/{len(BUYS)} tickers")
if failed:
    print(f"❌ Failed: {', '.join(failed)} — check Yahoo Finance ticker symbols")

## 5. Chart Style Configuration

In [ ]:
# Dark theme for all charts
plt.rcParams.update({
    "figure.facecolor": "#1a1a2e",
    "axes.facecolor":   "#16213e",
    "axes.edgecolor":   "#e0e0e0",
    "axes.labelcolor":  "#e0e0e0",
    "xtick.color":      "#e0e0e0",
    "ytick.color":      "#e0e0e0",
    "text.color":       "#e0e0e0",
    "grid.color":       "#2a2a4a",
    "grid.alpha":       0.5,
    "font.size":        10,
})

print("🎨 Chart style configured")

## 6. Per-Stock / ETF Charts

Each chart shows market price evolution with red triangle markers at buy points, annotated with shares × price.

In [ ]:
print("Generating per-stock charts …\n")

for name, df in prices.items():
    fig, ax = plt.subplots(figsize=(14, 6))

    # Price line
    ax.plot(df.index, df["price"], color="#00d4ff", linewidth=1.5,
            label="Market price", zorder=2)
    ax.fill_between(df.index, df["price"], alpha=0.15, color="#00d4ff")

    # Buy markers
    ccy = TICKER_CURRENCY.get(name, "")
    for date_str, num, price, costs in BUYS[name]:
        dt = pd.Timestamp(date_str)
        idx = df.index.get_indexer([dt], method="nearest")[0]
        if 0 <= idx < len(df):
            market_price = df["price"].iloc[idx]
            ax.scatter(df.index[idx], market_price,
                       color="#ff6b6b", s=120, zorder=5,
                       edgecolors="white", linewidths=1.2, marker="^")
            total = num * price
            ax.annotate(
                f"{num} × {price:.2f} {ccy}\n= {total:,.0f} {ccy}",
                xy=(df.index[idx], market_price),
                xytext=(15, 25), textcoords="offset points",
                fontsize=7.5, color="#ff6b6b",
                arrowprops=dict(arrowstyle="->", color="#ff6b6b", lw=0.8),
                bbox=dict(boxstyle="round,pad=0.3", fc="#1a1a2e",
                          ec="#ff6b6b", alpha=0.9),
                zorder=6,
            )

    ax.set_title(f"{name}  —  Market Price & Buy Points",
                 fontsize=16, fontweight="bold", pad=15)
    ax.set_xlabel("Date")
    ax.set_ylabel(f"Price ({ccy})")
    ax.legend(loc="upper left")
    ax.grid(True, linewidth=0.5)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    fig.autofmt_xdate()
    fig.tight_layout()

    fpath = os.path.join(OUT_DIR, f"{name}.png")
    fig.savefig(fpath, dpi=150, bbox_inches="tight")
    print(f"  ✅ {name}")
    plt.show()
    plt.close(fig)

print(f"\n🎉 All per-stock charts saved to {OUT_DIR}")

## 7. Combined Portfolio Chart

Shows total portfolio market value (green) vs. total invested / cost basis (orange dashed), with buy events marked.

In [ ]:
print("Generating combined portfolio chart …\n")

# ─── EUR/USD conversion ──────────────────────────────────
# Static approximation; for more accuracy you could pull
# historical EURUSD=X from yfinance
EUR_PER_USD = 0.88

# Build common date range
all_dates = pd.DatetimeIndex([])
for df in prices.values():
    all_dates = all_dates.union(df.index)
all_dates = all_dates.sort_values()

portfolio_value = pd.Series(0.0, index=all_dates)
total_invested = pd.Series(0.0, index=all_dates)
all_buy_events = []

for name, df in prices.items():
    buy_list = BUYS[name]
    ccy = TICKER_CURRENCY.get(name, "EUR")

    price_reindexed = df["price"].reindex(all_dates, method="ffill")
    cum_shares = compute_cumulative_shares(buy_list, all_dates)

    position_value = cum_shares * price_reindexed
    if ccy == "USD":
        position_value *= EUR_PER_USD

    portfolio_value += position_value.fillna(0)

    for date_str, num, price, costs in buy_list:
        dt = pd.Timestamp(date_str)
        amount = num * price + costs
        if ccy == "USD":
            amount *= EUR_PER_USD
        total_invested[total_invested.index >= dt] += amount
        all_buy_events.append((dt, name, num, amount))

# ─── Plot ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 8))

ax.plot(portfolio_value.index, portfolio_value.values,
        color="#00ff88", linewidth=2,
        label="Portfolio market value (EUR)", zorder=2)
ax.fill_between(portfolio_value.index, portfolio_value.values,
                alpha=0.15, color="#00ff88")

ax.plot(total_invested.index, total_invested.values,
        color="#ffaa00", linewidth=1.5, linestyle="--",
        label="Total invested (EUR)", zorder=2)

# Buy markers
for dt, name, num, amount_eur in all_buy_events:
    idx = portfolio_value.index.get_indexer([dt], method="nearest")[0]
    if 0 <= idx < len(portfolio_value):
        val = portfolio_value.iloc[idx]
        ax.scatter(portfolio_value.index[idx], val,
                   color="#ff6b6b", s=50, zorder=5, alpha=0.7,
                   edgecolors="white", linewidths=0.5, marker="^")

ax.set_title("Combined Portfolio — Market Value vs. Total Invested",
             fontsize=18, fontweight="bold", pad=15)
ax.set_xlabel("Date")
ax.set_ylabel("Value (EUR)")
ax.legend(loc="upper left", fontsize=12)
ax.grid(True, linewidth=0.5)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f"€{x:,.0f}"))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
fig.autofmt_xdate()
fig.tight_layout()

fpath = os.path.join(OUT_DIR, "COMBINED_PORTFOLIO.png")
fig.savefig(fpath, dpi=150, bbox_inches="tight")
print(f"  ✅ {fpath}")
plt.show()
plt.close(fig)

print(f"\n🎉 All done! Charts saved to {OUT_DIR}")

## 8. Portfolio Summary Table

In [ ]:
# Quick summary of current holdings
summary_rows = []

for name in BUYS:
    ccy = TICKER_CURRENCY.get(name, "EUR")
    total_shares = sum(num for _, num, _, _ in BUYS[name])
    total_cost = sum(num * price + costs for _, num, price, costs in BUYS[name])
    avg_price = sum(num * price for _, num, price, _ in BUYS[name]) / total_shares

    current_price = None
    if name in prices and len(prices[name]) > 0:
        current_price = prices[name]["price"].iloc[-1]

    row = {
        "Ticker": name,
        "Currency": ccy,
        "Shares": total_shares,
        "Avg Buy Price": round(avg_price, 2),
        "Total Cost": round(total_cost, 2),
    }

    if current_price is not None:
        row["Current Price"] = round(current_price, 2)
        row["Market Value"] = round(total_shares * current_price, 2)
        row["P/L"] = round(total_shares * current_price - total_cost, 2)
        row["P/L %"] = round((total_shares * current_price - total_cost) / total_cost * 100, 1)
    else:
        row["Current Price"] = "N/A"
        row["Market Value"] = "N/A"
        row["P/L"] = "N/A"
        row["P/L %"] = "N/A"

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df